# NHS England Waiting Times Analysis
**Author:** Drishti Kulkarni  
**Data:** NHS England RTT Waiting Times — March 2025  
**Tools:** Python (pandas, matplotlib, seaborn)

---

## What this notebook does
This notebook analyses NHS England Referral to Treatment (RTT) waiting times data to answer:
1. Which NHS trusts have the longest waiting times?
2. Which medical specialties have the worst backlogs?
3. What percentage of patients are waiting over 18 weeks (the NHS target)?
4. Which regions are performing worst?

---

## Section 1 — Import Libraries

In [ ]:
# Import all libraries we need
# pandas  = for loading and cleaning data (like Excel but in Python)
# matplotlib & seaborn = for creating charts
# warnings = to hide unnecessary warning messages

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set chart style — makes all charts look clean and professional
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✅ Libraries loaded successfully')

## Section 2 — Load the Data

**Important:** Make sure the CSV file is in your `data/` folder before running this cell.  
The file should be named: `20250331-RTT-March-2025-full-extract-revised.csv`

In [ ]:
# Load the NHS RTT data
# This file has ~185,000 rows — one row per NHS trust per specialty per pathway type

file_path = '../data/20250331-RTT-March-2025-full-extract-revised.csv'

print('Loading NHS data... (this may take 10-15 seconds due to file size)')
df = pd.read_csv(file_path, low_memory=False)

print(f'✅ Data loaded successfully!')
print(f'   Rows: {df.shape[0]:,}')
print(f'   Columns: {df.shape[1]}')

In [ ]:
# Preview the first 3 rows to understand what the data looks like
df.head(3)

In [ ]:
# Check the column names
print('Column names:')
for col in df.columns[:15]:  # Show first 15 columns
    print(f'  {col}')
print(f'  ... and {len(df.columns)-15} more week-band columns')

In [ ]:
# Check data types and missing values
print('Data types and missing values:')
print(df.dtypes[:15])
print(f'\nTotal missing values: {df.isnull().sum().sum():,}')

In [ ]:
# Understand what unique values exist in key columns
print(f"Unique NHS Trusts: {df['Provider Org Name'].nunique()}")
print(f"Unique Specialties: {df['Treatment Function Name'].nunique()}")
print(f"Unique Pathway Types: {df['RTT Part Description'].nunique()}")
print(f"\nPathway types available:")
for pt in df['RTT Part Description'].unique():
    print(f'  - {pt}')

## Section 3 — Clean the Data

The NHS data uses **week bands** — each column represents patients waiting in that week range.  
We need to:
1. Filter to the right pathway type (incomplete pathways = people still waiting)
2. Calculate total patients waiting over 18 weeks (the NHS target)
3. Clean up column names

In [ ]:
# Filter to 'Incomplete Pathways' — these are patients CURRENTLY waiting
# This is the most meaningful data for our analysis
df_waiting = df[df['RTT Part Description'] == 'Incomplete Pathways'].copy()

print(f'Rows after filtering to Incomplete Pathways: {df_waiting.shape[0]:,}')

# Also filter out 'Total' specialty rows to avoid double counting
df_waiting = df_waiting[df_waiting['Treatment Function Name'] != 'Total']
print(f'Rows after removing Total specialty rows: {df_waiting.shape[0]:,}')

In [ ]:
# Identify the week band columns
# These are all columns starting with 'Gt' (Greater than)
week_cols = [col for col in df.columns if col.startswith('Gt')]
print(f'Number of week band columns: {len(week_cols)}')
print(f'First 5: {week_cols[:5]}')
print(f'Last 3: {week_cols[-3:]}')

# Convert week columns to numeric (some may have been read as strings)
df_waiting[week_cols] = df_waiting[week_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

In [ ]:
# Calculate key metrics for each row

# Total patients waiting (using the 'Total All' column)
df_waiting['total_waiting'] = pd.to_numeric(df_waiting['Total All'], errors='coerce').fillna(0)

# Patients waiting OVER 18 weeks (columns Gt 18 To 19 onwards)
# The NHS target is that 92% of patients should be treated within 18 weeks
over_18_cols = [col for col in week_cols if any(
    f'Gt {i} To' in col or f'Gt {i} Weeks' in col 
    for i in range(18, 105)
)]

# Simpler approach — get all week columns from index 18 onwards
over_18_cols = week_cols[18:]  # columns from Gt 18 To 19 onwards
df_waiting['waiting_over_18wks'] = df_waiting[over_18_cols].sum(axis=1)

# Calculate percentage over 18 weeks
df_waiting['pct_over_18wks'] = (
    df_waiting['waiting_over_18wks'] / df_waiting['total_waiting'].replace(0, float('nan')) * 100
).fillna(0).round(1)

print('✅ Key metrics calculated:')
print(f'   Total patients waiting nationally: {df_waiting["total_waiting"].sum():,.0f}')
print(f'   Patients waiting over 18 weeks: {df_waiting["waiting_over_18wks"].sum():,.0f}')
total = df_waiting['total_waiting'].sum()
over18 = df_waiting['waiting_over_18wks'].sum()
print(f'   National % over 18 weeks: {over18/total*100:.1f}%')

In [ ]:
# Clean up provider names (remove NHS TRUST / NHS FOUNDATION TRUST suffix for cleaner charts)
df_waiting['Trust_Short'] = (
    df_waiting['Provider Org Name']
    .str.replace(' NHS FOUNDATION TRUST', '', regex=False)
    .str.replace(' NHS TRUST', '', regex=False)
    .str.title()
)

print('✅ Trust names cleaned')
print('Sample:', df_waiting['Trust_Short'].iloc[0])

In [ ]:
# Save the cleaned dataset for Tableau
# We'll save a summary version (not all 100+ week columns)

cols_to_save = [
    'Period', 'Provider Parent Name', 'Provider Org Code', 'Provider Org Name',
    'Trust_Short', 'Treatment Function Name', 'total_waiting',
    'waiting_over_18wks', 'pct_over_18wks'
]

df_clean = df_waiting[cols_to_save].copy()
df_clean.to_csv('../data/nhs_clean.csv', index=False)

print(f'✅ Clean data saved to data/nhs_clean.csv')
print(f'   Rows: {df_clean.shape[0]:,}')
print(f'   Columns: {df_clean.shape[1]}')
df_clean.head(3)

## Section 4 — Analysis & Visualisations

Now we answer our 4 key questions with charts.

### Question 1 — Which NHS Trusts have the most patients waiting?

In [ ]:
# Aggregate by Trust — sum all specialties per trust
trust_summary = df_waiting.groupby('Trust_Short').agg(
    total_waiting=('total_waiting', 'sum'),
    waiting_over_18wks=('waiting_over_18wks', 'sum')
).reset_index()

trust_summary['pct_over_18wks'] = (
    trust_summary['waiting_over_18wks'] / trust_summary['total_waiting'] * 100
).round(1)

# Top 15 trusts by total patients waiting
top_trusts = trust_summary.nlargest(15, 'total_waiting')

# Plot
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(top_trusts['Trust_Short'], top_trusts['total_waiting'], 
               color='#1f77b4', edgecolor='white')

# Add value labels
for bar, val in zip(bars, top_trusts['total_waiting']):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=9)

ax.set_xlabel('Total Patients Waiting', fontsize=12)
ax.set_title('Top 15 NHS Trusts by Total Patients Waiting\n(March 2025, All Specialties)', 
             fontsize=14, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../dashboard/chart1_top_trusts.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved')

### Question 2 — Which specialties have the worst backlogs?

In [ ]:
# Aggregate by specialty
specialty_summary = df_waiting.groupby('Treatment Function Name').agg(
    total_waiting=('total_waiting', 'sum'),
    waiting_over_18wks=('waiting_over_18wks', 'sum')
).reset_index()

specialty_summary['pct_over_18wks'] = (
    specialty_summary['waiting_over_18wks'] / specialty_summary['total_waiting'] * 100
).round(1)

# Top 12 specialties by total waiting
top_specialties = specialty_summary.nlargest(12, 'total_waiting')

# Plot — stacked bar showing within vs over 18 weeks
fig, ax = plt.subplots(figsize=(14, 7))

within_18 = top_specialties['total_waiting'] - top_specialties['waiting_over_18wks']
over_18 = top_specialties['waiting_over_18wks']

bars1 = ax.barh(top_specialties['Treatment Function Name'], within_18, 
                color='#2ca02c', label='Within 18 weeks ✓', edgecolor='white')
bars2 = ax.barh(top_specialties['Treatment Function Name'], over_18,
                left=within_18, color='#d62728', label='Over 18 weeks ✗', edgecolor='white')

ax.set_xlabel('Total Patients Waiting', fontsize=12)
ax.set_title('NHS Waiting List by Specialty — Within vs Over 18 Weeks\n(March 2025)', 
             fontsize=14, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../dashboard/chart2_specialties.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved')

### Question 3 — Which specialties have the highest % of patients over 18 weeks?

In [ ]:
# Filter to specialties with meaningful volume (500+ waiting) to avoid noise
meaningful = specialty_summary[specialty_summary['total_waiting'] >= 500]
worst_pct = meaningful.nlargest(12, 'pct_over_18wks')

# Colour bars: red if over 8% (breaching 92% target), amber otherwise
colors = ['#d62728' if p > 8 else '#ff7f0e' for p in worst_pct['pct_over_18wks']]

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(worst_pct['Treatment Function Name'], worst_pct['pct_over_18wks'],
               color=colors, edgecolor='white')

# Add % labels
for bar, val in zip(bars, worst_pct['pct_over_18wks']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

# Add NHS 8% target line (92% within 18 weeks = 8% over)
ax.axvline(x=8, color='black', linestyle='--', linewidth=1.5, label='NHS 8% threshold (92% target)')

ax.set_xlabel('% of Patients Waiting Over 18 Weeks', fontsize=12)
ax.set_title('Specialties With Highest % of Patients Waiting Over 18 Weeks\n(March 2025 — NHS England)', 
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../dashboard/chart3_pct_over18.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved')

### Question 4 — Which NHS Regions have the highest % over 18 weeks?

In [ ]:
# Use Provider Parent Name as a proxy for region/ICB
region_summary = df_waiting.groupby('Provider Parent Name').agg(
    total_waiting=('total_waiting', 'sum'),
    waiting_over_18wks=('waiting_over_18wks', 'sum')
).reset_index()

region_summary['pct_over_18wks'] = (
    region_summary['waiting_over_18wks'] / region_summary['total_waiting'] * 100
).round(1)

# Clean names for chart
region_summary['Region_Short'] = (
    region_summary['Provider Parent Name']
    .str.replace('NHS ', '', regex=False)
    .str.replace(' INTEGRATED CARE BOARD', '', regex=False)
    .str.replace(' INTEGRATED CARE SYSTEM', '', regex=False)
    .str.title()
    .str[:45]  # truncate long names
)

# Filter meaningful regions and sort
region_plot = region_summary[
    region_summary['total_waiting'] >= 1000
].nlargest(15, 'pct_over_18wks')

fig, ax = plt.subplots(figsize=(14, 8))
colors = ['#d62728' if p > 8 else '#1f77b4' for p in region_plot['pct_over_18wks']]
bars = ax.barh(region_plot['Region_Short'], region_plot['pct_over_18wks'],
               color=colors, edgecolor='white')

for bar, val in zip(bars, region_plot['pct_over_18wks']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

ax.axvline(x=8, color='black', linestyle='--', linewidth=1.5, label='NHS 8% threshold')
ax.set_xlabel('% of Patients Waiting Over 18 Weeks', fontsize=12)
ax.set_title('NHS Regions — % of Patients Waiting Over 18 Weeks\n(March 2025)', 
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../dashboard/chart4_regions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved')

## Section 5 — Summary of Key Findings

In [ ]:
# Print a clean summary of findings
total_w = df_waiting['total_waiting'].sum()
total_over18 = df_waiting['waiting_over_18wks'].sum()
pct_over18 = total_over18 / total_w * 100

top_trust = trust_summary.nlargest(1, 'total_waiting').iloc[0]
worst_trust_18 = trust_summary[trust_summary['total_waiting']>=1000].nlargest(1, 'pct_over_18wks').iloc[0]
worst_specialty = specialty_summary[specialty_summary['total_waiting']>=500].nlargest(1, 'pct_over_18wks').iloc[0]

print('=' * 60)
print('KEY FINDINGS — NHS WAITING TIMES ANALYSIS (MARCH 2025)')
print('=' * 60)
print(f'\n📊 NATIONAL PICTURE')
print(f'   Total patients on waiting list:  {total_w:>12,.0f}')
print(f'   Patients waiting over 18 weeks:  {total_over18:>12,.0f}')
print(f'   % waiting over 18 weeks:         {pct_over18:>11.1f}%')
print(f'   NHS target (should be below):    {8:>11}%')
print(f'\n🏥 TRUST WITH MOST PATIENTS WAITING')
print(f'   {top_trust["Trust_Short"]}')
print(f'   Total waiting: {top_trust["total_waiting"]:,.0f}')
print(f'\n⚠️  TRUST WITH HIGHEST % OVER 18 WEEKS')
print(f'   {worst_trust_18["Trust_Short"]}')
print(f'   % over 18 weeks: {worst_trust_18["pct_over_18wks"]}%')
print(f'\n🔬 SPECIALTY WITH HIGHEST % OVER 18 WEEKS')
print(f'   {worst_specialty["Treatment Function Name"]}')
print(f'   % over 18 weeks: {worst_specialty["pct_over_18wks"]}%')
print('=' * 60)

## Section 6 — Export Data for Tableau

The cleaned file `nhs_clean.csv` was already saved in Section 3.  
Use this file to build your Tableau dashboard.

**Columns available in nhs_clean.csv:**
- `Provider Org Name` / `Trust_Short` — NHS Trust
- `Provider Parent Name` — ICB / Region
- `Treatment Function Name` — Medical specialty
- `total_waiting` — Total patients waiting
- `waiting_over_18wks` — Patients waiting over 18 weeks
- `pct_over_18wks` — % waiting over 18 weeks

In [ ]:
print('✅ All done! Files saved:')
print('   data/nhs_clean.csv          → Use this in Tableau')
print('   dashboard/chart1_top_trusts.png')
print('   dashboard/chart2_specialties.png')
print('   dashboard/chart3_pct_over18.png')
print('   dashboard/chart4_regions.png')